Combine solar inverter power measurements, weather observations, and plant metadata into a single dataset for exploratory data analysis and solar power forecasting.

In [1]:
import pandas as pd

In [3]:
# Load processed datasets
inverter_clean = pd.read_csv(
    "../data/processed/PS_001_inverter_clean.csv",
    parse_dates=["datetime"]
)

weather_clean = pd.read_csv(
    "../data/processed/PS_001_weather_clean.csv",
    parse_dates=["datetime"]
)

metadata = pd.read_csv("../data/raw/BR-PVGen_metadata.csv")

In [5]:
plant_metadata = metadata[metadata['id'] == "PS_001"].copy()
plant_metadata

,brazil_federative_unit,id,is_panel_bifacial,nominal_power_mw,number_of_panels,panel_area_mm2,panel_bifaciality_coefficient,panel_efficiency_percentage,panel_temperature_coefficient,structure_type
0,SP,PS_001,True,5.0,12062,31516.36557,0.7,20.7,0.34,TRACKER


In [7]:
plant_power = (
    inverter_clean
    .groupby(["datetime", "ps_id"], as_index=False)
    .agg(
        total_active_power_w=(
            "total_active_power_w",
            lambda x: x.sum(min_count=1)
        ),
        reporting_inverters=(
            "total_active_power_w",
            "count"
        )
    )
)

In [8]:
#Merge inverter power with weather data
integrated_data = pd.merge(
    plant_power,
    weather_clean,
    on=["datetime", "ps_id"],
    how="inner",
    validate="one_to_one"
)

In [9]:
integrated_data["id"] = integrated_data["ps_id"]

integrated_data = integrated_data.merge(
    plant_metadata,
    left_on="ps_id",
    right_on="id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_metadata")
)

integrated_data = integrated_data.drop(columns=["id_metadata"])

In [10]:
integrated_data["hour"] = integrated_data["datetime"].dt.hour
integrated_data["day"] = integrated_data["datetime"].dt.day
integrated_data["month"] = integrated_data["datetime"].dt.month
integrated_data["day_of_week"] = (
    integrated_data["datetime"].dt.dayofweek
)

In [11]:
integrated_data = integrated_data.sort_values(
    "datetime"
).reset_index(drop=True)

In [12]:
integrated_data.head()

,datetime,ps_id,total_active_power_w,reporting_inverters,ghi_irradiance_wm2,poa_irradiance_wm2,ambient_temperature_celsius,panel_temperature_celsius,wind_speed_ms,id,...,number_of_panels,panel_area_mm2,panel_bifaciality_coefficient,panel_efficiency_percentage,panel_temperature_coefficient,structure_type,hour,day,month,day_of_week
0,2024-03-26 00:00:00+00:00,PS_001,NaN,0,1.825104,2.076329,22.69768,21.249123,1.954988,PS_001,...,12062,31516.36557,0.7,20.7,0.34,TRACKER,0,26,3,1
1,2024-03-26 00:15:00+00:00,PS_001,NaN,0,1.825104,2.076329,22.69768,21.249123,1.954988,PS_001,...,12062,31516.36557,0.7,20.7,0.34,TRACKER,0,26,3,1
2,2024-03-26 00:30:00+00:00,PS_001,NaN,0,1.825104,2.076329,22.69768,21.249123,1.954988,PS_001,...,12062,31516.36557,0.7,20.7,0.34,TRACKER,0,26,3,1
3,2024-03-26 00:45:00+00:00,PS_001,NaN,0,1.825104,2.076329,22.69768,21.249123,1.954988,PS_001,...,12062,31516.36557,0.7,20.7,0.34,TRACKER,0,26,3,1
4,2024-03-26 01:00:00+00:00,PS_001,NaN,0,1.825104,2.076329,22.69768,21.249123,1.954988,PS_001,...,12062,31516.36557,0.7,20.7,0.34,TRACKER,1,26,3,1


In [13]:
integrated_data.shape

(26431, 23)

In [14]:
integrated_data.columns.tolist()

['datetime',
 'ps_id',
 'total_active_power_w',
 'reporting_inverters',
 'ghi_irradiance_wm2',
 'poa_irradiance_wm2',
 'ambient_temperature_celsius',
 'panel_temperature_celsius',
 'wind_speed_ms',
 'id',
 'brazil_federative_unit',
 'is_panel_bifacial',
 'nominal_power_mw',
 'number_of_panels',
 'panel_area_mm2',
 'panel_bifaciality_coefficient',
 'panel_efficiency_percentage',
 'panel_temperature_coefficient',
 'structure_type',
 'hour',
 'day',
 'month',
 'day_of_week']

In [15]:
integrated_data.to_csv(
    "../data/processed/PS_001_integrated.csv",
    index=False
)